# 9. VaR Backtesting

This notebook evaluates the performance of Value at Risk (VaR) models by comparing predicted VaR thresholds with realized portfolio losses.

The analysis covers:

1. VaR Forecast Generation
2. VaR Exceptions
3. Exception Rate Analysis
4. Kupiec Proportion of Failures Test
5. VaR Model Comparison

The portfolio consists of SPY, QQQ, IWM, TLT, and GLD with fixed portfolio weights.

The objective is to assess whether the VaR models provide reliable estimates of portfolio downside risk by evaluating how frequently realized losses exceed the predicted VaR threshold.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from project_01_market_risk_engine.src.data import download_prices
from project_01_market_risk_engine.src.var import (
    historical_var,
    parametric_var,
    monte_carlo_var,
)

In [2]:
TICKERS = ["SPY", "QQQ", "IWM", "TLT", "GLD"]

START_DATE = "2015-01-01"
END_DATE = "2026-01-01"

weights = pd.Series(
    {
        "SPY": 0.30,
        "QQQ": 0.20,
        "IWM": 0.15,
        "TLT": 0.20,
        "GLD": 0.15,
    }
)

assert np.isclose(weights.sum(), 1.0), "Portfolio weights must sum to 1."

In [4]:
prices = download_prices(
    tickers=TICKERS,
    start=START_DATE,
    end=END_DATE,
)

returns = prices.pct_change().dropna()

portfolio_returns = returns @ weights

portfolio_returns.head()

[*********************100%***********************]  5 of 5 completed


Date
2015-01-05   -0.004954
2015-01-06   -0.002789
2015-01-07    0.006885
2015-01-08    0.008416
2015-01-09   -0.001263
dtype: float64

## 1. VaR Forecast Generation

VaR backtesting requires a sequence of out-of-sample VaR forecasts rather than a single VaR estimate calculated from the full historical sample.

For each forecast date, the VaR model is estimated using only returns available up to that date. The forecast is then compared with the subsequent realized portfolio return.

A rolling estimation window is used to maintain a consistent historical sample size for each forecast.

In [5]:
ROLLING_WINDOW = 500
CONFIDENCE_LEVELS = [0.95, 0.99]

backtest_returns = portfolio_returns.iloc[ROLLING_WINDOW:].copy()

backtest_returns.head()

Date
2016-12-28   -0.003696
2016-12-29    0.002462
2016-12-30   -0.004350
2017-01-03    0.006863
2017-01-04    0.006672
dtype: float64

In [27]:
var_columns = pd.MultiIndex.from_product(
    [
        ["Historical VaR", "Parametric VaR", "Monte Carlo VaR"],
        CONFIDENCE_LEVELS,
    ],
    names=["Model", "Confidence"],
)

var_forecasts = pd.DataFrame(
    index=backtest_returns.index,
    columns=var_columns,
    dtype=float,
)

In [28]:
var_forecasts.columns

MultiIndex([( 'Historical VaR', 0.95),
            ( 'Historical VaR', 0.99),
            ( 'Parametric VaR', 0.95),
            ( 'Parametric VaR', 0.99),
            ('Monte Carlo VaR', 0.95),
            ('Monte Carlo VaR', 0.99)],
           names=['Model', 'Confidence'])

In [29]:
for date in backtest_returns.index:
    window_returns = (portfolio_returns.loc[:date].iloc[-ROLLING_WINDOW-1:-1])

    var_forecasts.at[date, ("Historical VaR", 0.95)] = historical_var(
        returns=window_returns,
        confidence_level=0.95,
    )

    var_forecasts.at[date, ("Historical VaR", 0.99)] = historical_var(
        returns=window_returns,
        confidence_level=0.99,
    )

    var_forecasts.at[date, ("Parametric VaR", 0.95)] = parametric_var(
        returns=window_returns,
        confidence_level=0.95,
    )

    var_forecasts.at[date, ("Parametric VaR", 0.99)] = parametric_var(
        returns=window_returns,
        confidence_level=0.99,
    )

    window_asset_returns = returns.loc[window_returns.index]

    var_forecasts.at[date, ("Monte Carlo VaR", 0.95)] = monte_carlo_var(
        returns=window_asset_returns,
        weights=weights,
        confidence_level=0.95,
        n_simulations=10_000,
        random_seed=42,
    )

    var_forecasts.at[date, ("Monte Carlo VaR", 0.99)] = monte_carlo_var(
        returns=window_asset_returns,
        weights=weights,
        confidence_level=0.99,
        n_simulations=10_000,
        random_seed=42,
    )

In [30]:
var_forecasts.head()

Model      Historical VaR           Parametric VaR           Monte Carlo VaR  \
Confidence           0.95      0.99           0.95      0.99            0.95   
Date                                                                           
2016-12-28       0.009135  0.016495       0.009402  0.013391        0.009304   
2016-12-29       0.009135  0.016495       0.009396  0.013384        0.009296   
2016-12-30       0.009135  0.016495       0.009384  0.013372        0.009283   
2017-01-03       0.009135  0.016495       0.009400  0.013385        0.009295   
2017-01-04       0.009135  0.016495       0.009397  0.013379        0.009298   

Model                 
Confidence      0.99  
Date                  
2016-12-28  0.013046  
2016-12-29  0.013044  
2016-12-30  0.013049  
2017-01-03  0.013060  
2017-01-04  0.013041

## 2. VaR Exceptions

A VaR exception occurs when the realized portfolio loss exceeds the VaR forecast for the corresponding confidence level.

For example, at a 95% confidence level, approximately 5% of observations are expected to exceed the VaR threshold under a correctly specified model.

The exception indicator is defined as:

$$
I_t =
\begin{cases}
1, & L_t > VaR_t \\
0, & L_t \leq VaR_t
\end{cases}
$$

where $L_t$ represents the realized portfolio loss and $VaR_t$ represents the one-day VaR forecast.

In [31]:
realized_losses = -backtest_returns
realized_losses.name = "Realized Losses"
realized_losses.head()

Date
2016-12-28    0.003696
2016-12-29   -0.002462
2016-12-30    0.004350
2017-01-03   -0.006863
2017-01-04   -0.006672
Name: Realized Losses, dtype: float64

In [35]:
exceptions = pd.DataFrame(
    index=backtest_returns.index,
    columns=var_forecasts.columns,
    dtype=bool,
)

for column in var_forecasts.columns:
    exceptions[column] = (realized_losses > var_forecasts[column])

In [36]:
exceptions.sum()

Model            Confidence
Historical VaR   0.95          131
                 0.99           34
Parametric VaR   0.95          124
                 0.99           56
Monte Carlo VaR  0.95          126
                 0.99           58
dtype: int64

## 3. Exception Rate Analysis

The exception rate measures the proportion of backtesting observations in which realized losses exceeded the VaR forecast.

For a correctly specified VaR model, the observed exception rate should be approximately equal to the expected tail probability:

- 95% VaR → approximately 5% exceptions
- 99% VaR → approximately 1% exceptions

The exception rate is calculated as:

$$\text{Exception Rate} = \frac{\text{Number of Exceptions}}{\text{Number of Observations}}$$

Comparing observed and expected exception rates provides an initial assessment of VaR model calibration before applying the formal Kupiec Proportion of Failures test.

In [37]:
exception_rates = exceptions.mean()

exception_rates

Model            Confidence
Historical VaR   0.95          0.057837
                 0.99          0.015011
Parametric VaR   0.95          0.054746
                 0.99          0.024724
Monte Carlo VaR  0.95          0.055629
                 0.99          0.025607
dtype: float64

In [38]:
expected_rates = pd.Series(
    {
        0.95: 1 - 0.95,
        0.99: 1 - 0.99,
    }
)

expected_rates

0.95    0.05
0.99    0.01
dtype: float64

In [39]:
exception_rate_analysis = pd.DataFrame(
    {
        "Observed Rate": exception_rates,
        "Expected Rate": [
            expected_rates[level]
            for model, level in exception_rates.index
        ],
    },
    index=exception_rates.index,
)

exception_rate_analysis["Difference"] = (
    exception_rate_analysis["Observed Rate"]
    - exception_rate_analysis["Expected Rate"]
)

exception_rate_analysis

Observed Rate  Expected Rate  Difference
Model           Confidence                                          
Historical VaR  0.95             0.057837           0.05    0.007837
                0.99             0.015011           0.01    0.005011
Parametric VaR  0.95             0.054746           0.05    0.004746
                0.99             0.024724           0.01    0.014724
Monte Carlo VaR 0.95             0.055629           0.05    0.005629
                0.99             0.025607           0.01    0.015607

### Interpretation

The observed exception rates provide an initial indication of VaR model calibration.

At the 95% confidence level, all three models produce exception rates relatively close to the expected 5% level.

At the 99% confidence level, the observed exception rates are higher than the expected 1% level, particularly for the Parametric and Monte Carlo VaR models.

However, differences between observed and expected exception rates do not by themselves determine whether a VaR model is statistically inadequate. The Kupiec Proportion of Failures test formally evaluates whether the observed number of exceptions is consistent with the expected exception probability.

## 4. Kupiec Proportion of Failures Test

The Kupiec Proportion of Failures (POF) test formally evaluates whether the observed frequency of VaR exceptions is consistent with the expected exception probability.

The null hypothesis is:

$$H_0: p = p_0$$

where $p$ is the true probability of a VaR exception and $p_0 = 1-c$ is the expected exception probability at confidence level $c$.

The test statistic is:

$$LR_{POF} = -2 \ln \left[ \frac{(1-p_0)^{T-x}p_0^x}{(1-\hat{p})^{T-x}\hat{p}^x} \right]$$

where:

- $T$ is the total number of observations
- $x$ is the number of observed exceptions
- $p_0$ is the expected exception probability
- $\hat{p} = x/T$ is the observed exception rate

Under the null hypothesis, the model is accurate (the observed exception rate ${p}$ equals the model's expected exception rate $p_0$).

A p-value below the chosen significance level leads to rejection of the null hypothesis, indicating that the observed exception frequency is inconsistent with the expected VaR exception rate.

In [44]:
from scipy.stats import chi2


def kupiec_pof_test(
    exception_count: int,
    observation_count: int,
    confidence_level: float = 0.95,
    alpha: float = 0.05,
) -> tuple[float, bool]:
    """
    Perform the Kupiec Proportion of Failures test.

    Returns:
        tuple:
            p-value and whether the null hypothesis is rejected.
    """

    expected_rate = 1 - confidence_level
    observed_rate = exception_count / observation_count

    lr_statistic = -2 * (
        (observation_count - exception_count)
        * np.log((1 - expected_rate) / (1 - observed_rate))
        + exception_count
        * np.log(expected_rate / observed_rate)
    )

    p_value = 1 - chi2.cdf(lr_statistic, df=1)

    reject_null = p_value < alpha

    return p_value, reject_null

In [45]:
kupiec_results = []

for model, confidence in exceptions.columns:

    exception_count = exceptions[(model, confidence)].sum()

    p_value, reject_null = kupiec_pof_test(
        exception_count=exception_count,
        observation_count=len(exceptions),
        confidence_level=confidence,
    )

    kupiec_results.append({
        "Model": model,
        "Confidence": confidence,
        "p-value": p_value,
        "Reject H0": reject_null,
    })

kupiec_results = pd.DataFrame(kupiec_results)

kupiec_results

,Model,Confidence,p-value,Reject H0
0,Historical VaR,0.95,9.462308e-02,False
1,Historical VaR,0.99,2.565395e-02,True
2,Parametric VaR,0.95,3.070370e-01,False
3,Parametric VaR,0.99,3.005993e-09,True
4,Monte Carlo VaR,0.95,2.269138e-01,False
5,Monte Carlo VaR,0.99,4.385851e-10,True


### Interpretation

The Kupiec test evaluates whether the observed number of VaR exceptions is statistically consistent with the expected exception probability.

At the 5% significance level:

- **Fail to Reject H₀:** The observed exception frequency is statistically consistent with the expected VaR level.
- **Reject H₀:** The observed exception frequency is statistically inconsistent with the expected VaR level.

The test evaluates the frequency of exceptions only. It does not assess whether exceptions are independent or clustered over time.

## 5. VaR Model Comparison

The backtesting results are compared across the three VaR models using their exception rates and Kupiec test results.

A model is considered better calibrated when its exception rate is closer to the expected rate and the Kupiec test does not reject the null hypothesis.

In [47]:
model_comparison = kupiec_results.copy()

model_comparison["Exception Rate"] = [
    exceptions[(row["Model"], row["Confidence"])].mean()
    for _, row in model_comparison.iterrows()
]

model_comparison = model_comparison[
    [
        "Model",
        "Confidence",
        "Exception Rate",
        "p-value",
        "Reject H0",
    ]
]

model_comparison

,Model,Confidence,Exception Rate,p-value,Reject H0
0,Historical VaR,0.95,0.057837,9.462308e-02,False
1,Historical VaR,0.99,0.015011,2.565395e-02,True
2,Parametric VaR,0.95,0.054746,3.070370e-01,False
3,Parametric VaR,0.99,0.024724,3.005993e-09,True
4,Monte Carlo VaR,0.95,0.055629,2.269138e-01,False
5,Monte Carlo VaR,0.99,0.025607,4.385851e-10,True


The results show that all three VaR models provide acceptable coverage at the 95% confidence level, while all three models are rejected at the 99% confidence level.

The higher exception rates at the 99% confidence level indicate that the models underestimate the frequency of extreme portfolio losses in the backtesting period.

## Key Observations

- VaR backtesting compares predicted risk thresholds with realized portfolio losses.
- All three models produce exception rates reasonably close to the expected 5% level at 95% confidence.
- All three models produce higher-than-expected exception rates at 99% confidence.
- The Kupiec test does not reject the models at 95% confidence but rejects them at 99%.
- The results highlight the difficulty of accurately estimating extreme-tail losses using VaR models.